# Feedforward Neural Networks

In [ ]:
# Optional dependency installer for local Jupyter use
import sys
import subprocess
import importlib.util

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "gymnasium": "gymnasium[classic-control]",
    "pygame": "pygame",
    "matplotlib": "matplotlib",
}

for module_name, pip_name in required.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

print("Dependency check complete.")


In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import gymnasium as gym
import pygame
import matplotlib.pyplot as plt

torch.set_printoptions(precision=6, sci_mode=False)
np.set_printoptions(suppress=True)


## Task 0

In [ ]:
class CartPoleHumanRecorder:
    def __init__(self, n_episodes=30, base_dir="cartpole_human_data"):
        self.n_episodes = n_episodes
        self.base_dir = Path(base_dir)
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        self.output_dir = self.base_dir / f"run_{timestamp}"
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.env = gym.make("CartPole-v1", render_mode="human")

        self.states = []
        self.actions = []
        self.episode_rewards = []
        self.episode_lengths = []

        pygame.init()
        pygame.display.set_caption("CartPole Human Control")
        self.closed = False

    def _get_action_blocking(self):
        while True:
            event = pygame.event.wait()

            if event.type == pygame.QUIT:
                self.closed = True
                return None

            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_LEFT:
                    return 0
                if event.key == pygame.K_RIGHT:
                    return 1

    def play(self):
        print(f"Saving outputs to: {self.output_dir}")

        for ep in range(self.n_episodes):
            state, info = self.env.reset()
            terminated = False
            truncated = False
            total_reward = 0.0
            steps = 0

            while not (terminated or truncated):
                action = self._get_action_blocking()

                if action is None:
                    print("Window closed. Stopping early.")
                    self._save()
                    self.env.close()
                    pygame.quit()
                    return

                self.states.append(state.copy())
                self.actions.append(action)

                next_state, reward, terminated, truncated, info = self.env.step(action)

                total_reward += reward
                steps += 1
                state = next_state

            self.episode_rewards.append(total_reward)
            self.episode_lengths.append(steps)

            print(f"Episode {ep + 1}/{self.n_episodes} | reward={total_reward:.0f} | steps={steps}")

        self._save()
        self.env.close()
        pygame.quit()

    def _save(self):
        states_arr = np.array(self.states, dtype=np.float32)
        actions_arr = np.array(self.actions, dtype=np.int64)

        if len(states_arr) == 0:
            print("No samples recorded; nothing to save.")
            return

        data_df = pd.DataFrame(states_arr, columns=["x", "x_dot", "theta", "theta_dot"])
        data_df["action"] = actions_arr
        data_df.to_csv(self.output_dir / "cartpole_data.csv", index=False)

        metrics_df = pd.DataFrame({
            "episode": np.arange(1, len(self.episode_rewards) + 1),
            "reward": self.episode_rewards,
            "steps": self.episode_lengths,
        })
        metrics_df.to_csv(self.output_dir / "episode_metrics.csv", index=False)

        np.save(self.output_dir / "states.npy", states_arr)
        np.save(self.output_dir / "actions.npy", actions_arr)

        print("\nSaved files:")
        print(self.output_dir / "cartpole_data.csv")
        print(self.output_dir / "episode_metrics.csv")

        print("\nSummary statistics")
        print(f"Samples recorded: {len(actions_arr)}")
        print(f"Average reward: {np.mean(self.episode_rewards):.2f}")
        print(f"Max reward: {np.max(self.episode_rewards):.2f}")
        print(f"Min reward: {np.min(self.episode_rewards):.2f}")
        print(f"Average steps: {np.mean(self.episode_lengths):.2f}")
        print(f"Max steps: {np.max(self.episode_lengths):.2f}")
        print(f"Min steps: {np.min(self.episode_lengths):.2f}")


## Task 1: Run the recorder

Run the next cell locally to collect 20 episodes of human demonstrations. Use the left and right arrow keys to control the cart.

After it finishes:
- a timestamped folder should be created,
- `cartpole_data.csv` and `episode_metrics.csv` should be present,
- the number of rows in `cartpole_data.csv` should equal the total number of time steps across all episodes.


In [ ]:
# Assignment-required script block
if __name__ == "__main__":
    rec = CartPoleHumanRecorder(n_episodes=20)
    rec.play()


### After running the recorder

Set `RUN_DIR` below to the timestamped folder created on your machine. Example:

```python
RUN_DIR = Path("cartpole_human_data/run_20260421_153000")
```


In [ ]:
from pathlib import Path

RUN_DIR = Path("cartpole_human_data/run_20260421_013849")
print("Current RUN_DIR =", RUN_DIR)


## Confirm that the saved data are consistent

This verifies that:
- the required files exist,
- the number of recorded samples matches total time steps across episodes,
- the saved dimensions are correct.


In [ ]:
data_path = RUN_DIR / "cartpole_data.csv"
metrics_path = RUN_DIR / "episode_metrics.csv"

if not data_path.exists() or not metrics_path.exists():
    raise FileNotFoundError("Set RUN_DIR to your actual timestamped folder before running this section.")

df = pd.read_csv(data_path)
metrics_df = pd.read_csv(metrics_path)

print("cartpole_data.csv shape:", df.shape)
print(df.head())
print()
print("episode_metrics.csv shape:", metrics_df.shape)
print(metrics_df.head())
print()

n_samples = len(df)
total_steps = metrics_df["steps"].sum()
print("Recorded samples:", n_samples)
print("Total steps across episodes:", total_steps)
print("Match:", n_samples == total_steps)


## Task 2: Build the supervised learning dataset

We learn a mapping from state in `R^4` to action in `{0, 1}`.

The next cell loads the dataset, builds features and labels, and splits the data into train, validation, and test sets.


In [ ]:
X = df[["x", "x_dot", "theta", "theta_dot"]].to_numpy(dtype=np.float32)
y = df["action"].to_numpy(dtype=np.int64)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("X_train shape:", X_train.shape, "| y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape, "| y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape, "| y_test shape:", y_test.shape)


## Task 3: Normalize the features

Normalization is based only on training-set statistics. The same mean and standard deviation are then used for validation and test sets.


In [ ]:
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)
train_std = np.where(train_std == 0, 1.0, train_std)

X_train_norm = (X_train - train_mean) / train_std
X_val_norm = (X_val - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std

print("Training mean:", train_mean)
print("Training std:", train_std)
print()
print("Normalized training-set means (approx):", X_train_norm.mean(axis=0))
print("Normalized training-set stds (approx):", X_train_norm.std(axis=0))


## Task 4: Construct DataLoaders

Use batch size 64 and shuffle only the training data.


In [ ]:
X_train_t = torch.tensor(X_train_norm, dtype=torch.float32)
X_val_t = torch.tensor(X_val_norm, dtype=torch.float32)
X_test_t = torch.tensor(X_test_norm, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


The model shows steady improvement during training. Training accuracy increases from approximately 0.56 to around 0.74, while validation accuracy stabilizes around 0.75. At the same time, both training and validation loss consistently decrease, indicating effective learning. The small gap between training and validation accuracy suggests that the model generalizes well and does not suffer from significant overfitting.

## Task 5: Define the neural network

The model maps `R^4` to `R^2` and outputs logits for the two actions. It uses two hidden layers with ReLU activations.


In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=64, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)

model = PolicyNet()
print(model)


The model achieves a test accuracy of 0.7705, which is consistent with the validation accuracy. This indicates that the model generalizes well to unseen data and has learned a meaningful mapping from states to actions.

## Task 6: Train the model

Train with:
- cross-entropy loss,
- Adam optimizer.

At each epoch, report training loss, training accuracy, validation loss, and validation accuracy.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == yb).sum().item()
            total_count += xb.size(0)

    return total_loss / total_count, total_correct / total_count

n_epochs = 30
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, n_epochs + 1):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_count = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == yb).sum().item()
        running_count += xb.size(0)

    train_loss = running_loss / running_count
    train_acc = running_correct / running_count
    val_loss, val_acc = evaluate_loader(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history["train_acc"], label="train_acc")
plt.plot(history["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.show()


The training and validation loss curves both decrease smoothly and converge over time. There is no divergence between the two curves, which indicates stable training and minimal overfitting. The validation loss remains slightly lower than training loss in later epochs, suggesting that the model maintains good generalization performance.

## Task 7: Evaluate on the test set

This measures how well the learned policy generalizes to unseen state-action pairs.


In [ ]:
test_loss, test_acc = evaluate_loader(model, test_loader, criterion, device)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


Both training and validation accuracy increase steadily and plateau around 0.74–0.76. The close alignment between the two curves indicates that the model is neither underfitting nor overfitting, but instead achieving a balanced level of performance.

## Task 8: Evaluate the learned policy in the environment

At each time step:
1. normalize the current state using training-set mean and standard deviation,
2. compute model logits,
3. choose `argmax(logits)` as the action.

We then compare the learned policy's average reward with the recorded human average reward.


In [ ]:
def normalize_state(state, mean, std):
    state = np.asarray(state, dtype=np.float32)
    return (state - mean) / std

def run_policy_episode(model, env, mean, std, device):
    state, info = env.reset()
    terminated = False
    truncated = False
    total_reward = 0.0
    steps = 0

    while not (terminated or truncated):
        state_norm = normalize_state(state, mean, std)
        state_t = torch.tensor(state_norm, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(state_t)
            action = int(torch.argmax(logits, dim=1).item())

        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1

    return total_reward, steps

def evaluate_policy(model, n_episodes, mean, std, device, render_mode=None):
    env = gym.make("CartPole-v1", render_mode=render_mode)
    rewards = []
    lengths = []

    for ep in range(n_episodes):
        reward, steps = run_policy_episode(model, env, mean, std, device)
        rewards.append(reward)
        lengths.append(steps)

    env.close()
    return rewards, lengths

policy_rewards, policy_lengths = evaluate_policy(
    model=model,
    n_episodes=20,
    mean=train_mean,
    std=train_std,
    device=device,
    render_mode=None,
)

human_avg = metrics_df["reward"].mean()
policy_avg = float(np.mean(policy_rewards))

print("Human average reward:", human_avg)
print("Policy average reward:", policy_avg)
print("Policy rewards:", policy_rewards)


The learned policy achieved an average reward of approximately 9.85, compared with my average human reward of 20.3. This indicates that the imitation model underperformed the demonstrated policy in the environment.

## Task 9: Visual evaluation with rendering enabled

Run the next cell locally because it opens a GUI window. It runs 5 episodes and reports episode rewards and summary statistics.


In [ ]:
visual_rewards, visual_lengths = evaluate_policy(
    model=model,
    n_episodes=5,
    mean=train_mean,
    std=train_std,
    device=device,
    render_mode="human",
)

visual_df = pd.DataFrame({
    "episode": np.arange(1, 6),
    "reward": visual_rewards,
    "steps": visual_lengths,
})

print(visual_df)
print()
print("Average reward:", visual_df["reward"].mean())
print("Maximum reward:", visual_df["reward"].max())
print("Minimum reward:", visual_df["reward"].min())


Visually, the learned policy was somewhat unstable. It was able to keep the pole balanced for short durations but often failed to recover once the pole deviated significantly. The agent tended to overcorrect, suggesting that the model captured some local decision patterns but did not fully learn stable long-horizon control behavior.

## Task 10: Final reflection

This experiment demonstrates that imitation learning can learn useful local policies from labeled data, especially in low-dimensional environments like CartPole. However, it does not optimize long-term rewards, which leads to weaker performance in sequential tasks. Errors accumulate over time, causing performance to degrade compared to human control.